## 1. Choix de l'Ansatz : UCCSD vs Hardware Efficient (HEA)

Dans cette première étape, nous allons étudier une chaîne d'hydrogène ($H_3$ ou $H_4$). La structure de l'Ansatz (le circuit quantique paramétré) est déterminante. Nous allons comparer un Ansatz chimiquement inspiré, le **UCCSD**, avec un Ansatz agnostique conçu pour minimiser la profondeur sur les puces réelles, le **Hardware Efficient Ansatz (HEA)**.

**Méthodologie détaillée :**
1. Définissez la géométrie de votre molécule ($H_3$ linéaire avec un espacement de ~0.735 Å).
2. Utilisez `PySCFDriver` pour extraire le problème électronique, puis `ParityMapper` pour obtenir l'Hamiltonien sous forme de matrices de Pauli.
3. Préparez l'état initial de Hartree-Fock.
4. Générez le circuit UCCSD via la bibliothèque Qiskit Nature.
5. Générez le circuit HEA (Hardware Efficient Ansatz) en utilisant la classe `TwoLocal` de Qiskit (privilégiez des rotations $R_y$ et des intrications $CX$, avec `reps=2`).
6. Décomposez vos circuits (via `.decompose()`) pour évaluer leurs ressources natives. Affichez la profondeur totale et le nombre de portes CNOT pour comparer l'encombrement matériel des deux approches.

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit.circuit.library import TwoLocal

# Initialisation de la molécule et de l'Hamiltonien qubit
driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735; H 0 0 1.47", ...) # A compléter
problem = driver.run()
mapper = ParityMapper(...) # A compléter
qubit_op = ... # A compléter

# Création de l'état initial (Hartree-Fock)
init_state = HartreeFock(...) # A compléter

# Construction de l'Ansatz UCCSD
ansatz_uccsd = UCCSD(...) # A compléter
ansatz_uccsd.compose(init_state, front=True, inplace=True)

# Construction du Hardware Efficient Ansatz (HEA)
ansatz_hea = TwoLocal(...) # A compléter
ansatz_hea.compose(init_state, front=True, inplace=True)

# Décomposition et évaluation des métriques matérielles
circuit_uccsd_dec = ansatz_uccsd.decompose().decompose()
circuit_hea_dec = ansatz_hea.decompose()

depth_uccsd = ... # A compléter avec la bonne méthode
cnots_uccsd = ... # A compléter (indice: count_ops)
depth_hea = ... # A compléter
cnots_hea = ... # A compléter

print(f"UCCSD -> Profondeur: {depth_uccsd}, CNOTs: {cnots_uccsd}")
print(f"HEA   -> Profondeur: {depth_hea}, CNOTs: {cnots_hea}")


## 2. Implémentation manuelle du VQE et comparaison des optimiseurs

Plutôt que d'utiliser la classe `VQE` de Qiskit, vous allez coder la boucle d'optimisation. Cela implique de créer une **fonction de coût** qui fait le lien entre les paramètres classiques et l'évaluation quantique de l'énergie.

Nous comparerons deux optimiseurs classiques :
* **COBYLA** (via `scipy.optimize`) : Efficace, mais sensible au bruit.
* **SPSA** (via `qiskit_algorithms.optimizers`) : Approche stochastique, résiliente au bruit.

**Méthodologie détaillée :**
1. Instanciez la primitive `Estimator`.
2. Créez une fonction `create_cost_function` qui encapsule l'évaluation de l'énergie et enregistre les valeurs dans une liste.
3. Définissez un point de départ aléatoire pour vos paramètres.
4. Lancez les optimisations pour les paires (UCCSD, COBYLA), (UCCSD, SPSA), (HEA, COBYLA) et (HEA, SPSA).
5. Tracez les 4 courbes de convergence.

In [ ]:
import numpy as np
from qiskit.primitives import Estimator
from scipy.optimize import minimize
from qiskit_algorithms.optimizers import SPSA
import matplotlib.pyplot as plt

estimator = Estimator()

# Historiques pour stocker l'énergie à chaque itération
history_uccsd_cobyla, history_uccsd_spsa = [], []
history_hea_cobyla, history_hea_spsa = [], []

# Fonction de coût générique
def create_cost_function(ansatz, history_list):
    def cost_function(params):
        # A compléter : utilisez estimator.run(...) pour évaluer l'énergie
        job = ... 
        energy = job.result().values[0]
        history_list.append(energy)
        return energy
    return cost_function

# --- Optimisation UCCSD ---
initial_params_uccsd = np.random.uniform(-np.pi, np.pi, ansatz_uccsd.num_parameters)

# UCCSD + COBYLA
cost_uccsd_cobyla = create_cost_function(ansatz_uccsd, history_uccsd_cobyla)
res_uccsd_cobyla = minimize(cost_uccsd_cobyla, initial_params_uccsd, method='COBYLA')

# UCCSD + SPSA
spsa_optimizer = SPSA(maxiter=150)
cost_uccsd_spsa = create_cost_function(ansatz_uccsd, history_uccsd_spsa)
# A compléter : utilisez spsa_optimizer.minimize(...)
res_uccsd_spsa = ... 

# --- Optimisation HEA ---
initial_params_hea = np.random.uniform(-np.pi, np.pi, ansatz_hea.num_parameters)

# A compléter : Lancez l'optimisation pour HEA + COBYLA et HEA + SPSA
# cost_hea_cobyla = ...
# res_hea_cobyla = ...
# cost_hea_spsa = ...
# res_hea_spsa = ...

# Tracé des courbes de convergence
plt.figure(figsize=(10, 6))
plt.plot(history_uccsd_cobyla, label='UCCSD + COBYLA')
plt.plot(history_uccsd_spsa, label='UCCSD + SPSA')
plt.plot(history_hea_cobyla, label='HEA + COBYLA')
plt.plot(history_hea_spsa, label='HEA + SPSA')
plt.xlabel('Itérations')
plt.ylabel('Énergie (Hartree)')
plt.title('Comparaison de la convergence des Ansatz et Optimiseurs')
plt.legend()
plt.grid(True)
plt.show()

## 3. Le phénomène des Barren Plateaus (Plateaux arides)

Pourquoi ne pas toujours utiliser un Ansatz HEA très profond ? Plus le circuit est profond et intriqué, plus le paysage d'optimisation s'aplatit : les gradients s'évanouissent de manière exponentielle, rendant l'optimisation impossible.

**Méthodologie détaillée :**
1. Utilisez `ParamShiftEstimatorGradient` de Qiskit.
2. Faites varier la profondeur de votre Ansatz HEA (`reps = 1, 3, 5, 8`).
3. Pour chaque profondeur, échantillonnez 50 jeux de paramètres aléatoires.
4. Calculez le gradient de l'énergie par rapport au *premier* paramètre $\theta_0$.
5. Calculez la **variance** de ces 50 gradients.
6. Tracez la variance en fonction de la profondeur.

In [ ]:
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

gradient_calc = ParamShiftEstimatorGradient(estimator)
profondeurs = [1, 3, 5, 8]
variances = []
num_samples = 50

for p in profondeurs:
    # A compléter : créez un TwoLocal avec la profondeur p
    ansatz_bp = ... 
    ansatz_bp.compose(init_state, front=True, inplace=True)
    
    grads_theta0 = []
    for _ in range(num_samples):
        params = np.random.uniform(-np.pi, np.pi, ansatz_bp.num_parameters)
        
        # A compléter : utilisez gradient_calc.run(...) pour évaluer le gradient
        grad_result = ... 
        
        # Extraction du gradient par rapport au paramètre 0
        valeur_grad_theta0 = grad_result.gradients[0][0]
        grads_theta0.append(valeur_grad_theta0)
        
    variance_p = np.var(grads_theta0)
    variances.append(variance_p)
    print(f"Profondeur {p} : Variance = {variance_p:.2e}")

# Tracé de la variance en échelle logarithmique
plt.figure(figsize=(8, 5))
plt.plot(profondeurs, variances, marker='o', linestyle='-', color='b')
plt.yscale('log')
plt.xlabel('Profondeur de l\'Ansatz (reps)')
plt.ylabel('Variance du Gradient (Échelle Log)')
plt.title('Démonstration du phénomène des Barren Plateaus (HEA)')
plt.grid(True, which="both", ls="--")
plt.show()

# ==========================================
# PARTIE 2 : TP ADAPT-VQE (Construction dynamique)
# ==========================================


## 1. Implémentation manuelle de l'ADAPT-VQE

L'ADAPT-VQE résout le problème des plateaux arides en construisant le circuit itérativement. Plutôt qu'un Ansatz fixe, on dispose d'un "pool" d'opérateurs. À chaque étape, on ajoute l'opérateur qui offre la plus forte pente de descente de l'énergie.

**Méthodologie détaillée :**
1. **Pool d'opérateurs :** Créez une liste contenant les opérateurs de Pauli pertinents (ex: extraits des excitations fermioniques).
2. **Boucle adaptative :** À chaque itération :
   - Pour *chaque* opérateur du pool, estimez le gradient (à paramètre $\theta=0$).
   - Identifiez le gradient absolu maximum. S'il est sous le `gradient_threshold` (1e-3), arrêtez.
   - Ajoutez l'opérateur correspondant à votre circuit courant (via `PauliEvolutionGate`).
   - Ré-optimisez *tous* les paramètres du nouveau circuit en réutilisant la fonction `create_cost_function` de la partie 1.

In [ ]:
from qiskit.circuit.library import PauliEvolutionGate

# Construction du pool d'opérateurs (Astuce: créez une liste d'opérateurs de Pauli)
operator_pool = ... # A compléter

current_circuit = init_state.copy()
history_adapt = []
gradient_threshold = 1e-3

for iteration in range(20): # Limite de sécurité
    print(f"--- Cycle ADAPT-VQE {iteration+1} ---")
    
    # 1. Évaluation des gradients du pool
    gradients = []
    for op in operator_pool:
        # A compléter : évaluez le gradient pour cet opérateur
        grad_val = ... 
        gradients.append(np.abs(grad_val))
        
    # 2. Sélection du meilleur opérateur
    max_grad_idx = np.argmax(gradients)
    max_grad = gradients[max_grad_idx]
    best_op = operator_pool[max_grad_idx]
    
    print(f"Gradient Max : {max_grad:.4f}")
    if max_grad < gradient_threshold:
        print("Convergence atteinte (Gradient sous le seuil).")
        break
        
    # 3. Ajout au circuit
    # A compléter : utilisez PauliEvolutionGate pour ajouter best_op au circuit
    # gate = PauliEvolutionGate(...)
    # current_circuit.append(...)
    
    # 4. Ré-optimisation complète
    initial_params = np.zeros(current_circuit.num_parameters) 
    cost_func_adapt = create_cost_function(current_circuit, []) # Historique vide localement
    
    # A compléter : utilisez scipy.minimize (ex: COBYLA)
    res_adapt = minimize(...) 
    
    current_energy = res_adapt.fun
    history_adapt.append(current_energy)
    print(f"Énergie après cycle : {current_energy:.4f}\n")

## 2. Variante : k-ADAPT-VQE

Une limite de l'ADAPT-VQE standard est l'ajout d'un seul opérateur à la fois, ce qui demande de très nombreuses étapes de ré-optimisation coûteuses. L'approche **k-ADAPT-VQE** propose d'ajouter les $k$ meilleurs opérateurs d'un coup.

**Méthodologie détaillée :**
1. Dupliquez votre logique ADAPT-VQE.
2. Au lieu de sélectionner le maximum, utilisez `np.argsort` pour trouver les **$k$ opérateurs** ayant les plus grands gradients absolus (ex: $k=2$).
3. Ajoutez ces $k$ opérateurs successivement à votre circuit.
4. Ré-optimisez l'ensemble.

In [ ]:
k = 2
current_circuit_k = init_state.copy()
history_k_adapt = []

for iteration in range(20):
    print(f"--- Cycle {k}-ADAPT-VQE {iteration+1} ---")
    
    # 1. Évaluation des gradients
    gradients = []
    for op in operator_pool:
        # A compléter : même calcul de gradient qu'à la cellule 8
        grad_val = ... 
        gradients.append(np.abs(grad_val))
        
    # 2. Sélection des k meilleurs opérateurs
    indices_tries = np.argsort(gradients)[::-1]
    top_k_indices = indices_tries[:k]
    
    print(f"Top {k} Gradients : {[gradients[i] for i in top_k_indices]}")
    if gradients[top_k_indices[0]] < gradient_threshold:
         print("Convergence atteinte.")
         break
         
    # 3. Ajout groupé au circuit
    for idx in top_k_indices:
        op_to_add = operator_pool[idx]
        # A compléter : ajouter l'opérateur au current_circuit_k
        # ...
        
    # 4. Ré-optimisation complète
    initial_params = np.zeros(current_circuit_k.num_parameters)
    cost_func_k = create_cost_function(current_circuit_k, [])
    
    # A compléter : lancer l'optimisation
    res_k_adapt = minimize(...) 
    
    current_energy_k = res_k_adapt.fun
    history_k_adapt.append(current_energy_k)
    print(f"Énergie après ajout groupé : {current_energy_k:.4f}\n")

## 3. Comparaison Globale : VQE vs ADAPT-VQE vs k-ADAPT-VQE

Pour conclure, visualisons les bénéfices des implémentations adaptatives face au VQE standard, tant en termes de vitesse de convergence (par cycle) que de ressources (profondeur).

**Méthodologie détaillée :**
1. **Énergie :** Tracez sur un même graphique l'historique complet du VQE (UCCSD) du TP1 et les paliers d'énergie atteints à chaque cycle majeur par l'ADAPT-VQE et le k-ADAPT-VQE.
2. **Ressources :** Décomposez les circuits finaux générés et comparez leur profondeur et le nombre de CNOTs par rapport au UCCSD.


In [ ]:
# Tracé comparatif des historiques de convergence
plt.figure(figsize=(10, 6))

# VQE Standard (On reprend l'historique de la Partie 1)
plt.plot(history_uccsd_cobyla, label='VQE Standard (UCCSD)', color='blue', alpha=0.6)

# ADAPT-VQE (Paliers par cycle)
plt.plot(range(1, len(history_adapt) + 1), history_adapt, 
         marker='o', linestyle='-', color='red', label='ADAPT-VQE (k=1)')
         
# k-ADAPT-VQE (Paliers par cycle)
plt.plot(range(1, len(history_k_adapt) + 1), history_k_adapt, 
         marker='s', linestyle='-', color='green', label=f'k-ADAPT-VQE (k={k})')

plt.xlabel('Itérations VQE / Cycles ADAPT')
plt.ylabel('Énergie (Hartree)')
plt.title('Dynamique de convergence : VQE vs ADAPT-VQE vs k-ADAPT-VQE')
plt.legend()
plt.grid(True)
plt.show()

# Évaluation des métriques matérielles finales
circuit_adapt_dec = current_circuit.decompose().decompose()
depth_adapt = ... # A compléter
cnots_adapt = ... # A compléter

circuit_k_adapt_dec = current_circuit_k.decompose().decompose()
depth_k_adapt = ... # A compléter
cnots_k_adapt = ... # A compléter

print("\n--- Bilan des Ressources Matérielles Finales ---")
print(f"VQE Standard (UCCSD) -> Profondeur: {depth_uccsd}, CNOTs: {cnots_uccsd}") # Variable de la cellule 2
print(f"ADAPT-VQE (k=1)      -> Profondeur: {depth_adapt}, CNOTs: {cnots_adapt}")
print(f"k-ADAPT-VQE (k={k})    -> Profondeur: {depth_k_adapt}, CNOTs: {cnots_k_adapt}")

# ==========================================
# PARTIE 3 : TP Greedy-VQE (Construction dynamique en gelant les paramètres)
# ==========================================

## 1. Variante : ADAPT-VQE avec Optimisation Séquentielle (Paramètres gelés)

Dans l'ADAPT-VQE standard, à chaque fois que nous ajoutons une porte, nous ré-optimisons **l'intégralité** des paramètres du circuit depuis le début. Cela coûte très cher en termes de ressources classiques (appels à l'optimiseur) et quantiques (évaluations du circuit).

Une approche alternative, plus "gloutonne" (greedy), consiste à **geler** les paramètres précédemment trouvés. À chaque étape, nous n'optimisons **que** le paramètre de la porte qui vient d'être ajoutée.

**Méthodologie détaillée :**

1. Reprenez la boucle de l'ADAPT-VQE de la Partie 1 (sélection du meilleur opérateur via les gradients).
2. Maintenez une liste `fixed_params` qui stocke la valeur finale des paramètres des itérations précédentes.
3. Créez une nouvelle fonction de coût locale `cost_sequential` : elle ne prend en entrée qu'un seul paramètre (le nouveau), le concatène avec `fixed_params`, et évalue l'énergie globale.
4. Lancez `scipy.minimize` sur cet unique paramètre.
5. Ajoutez la valeur optimisée trouvée à `fixed_params` pour le cycle suivant.

In [ ]:
import numpy as np
from scipy.optimize import minimize

current_circuit_seq = init_state.copy()
history_seq_adapt = []
fixed_params = [] # Stockage des paramètres gelés

for iteration in range(20):
    print(f"--- Cycle Séq-ADAPT-VQE {iteration+1} ---")
    
    # 1 & 2. Évaluation des gradients et sélection (Identique à ADAPT-VQE standard)
    gradients = []
    for op in operator_pool:
        # A compléter : calcul du gradient 
        grad_val = ... 
        gradients.append(np.abs(grad_val))
        
    max_grad_idx = np.argmax(gradients)
    best_op = operator_pool[max_grad_idx]
    
    if gradients[max_grad_idx] < gradient_threshold:
        print("Convergence atteinte.")
        break
        
    # 3. Ajout au circuit
    # A compléter : utilisez PauliEvolutionGate
    # current_circuit_seq.append(...)
    
    # 4. Optimisation Séquentielle (uniquement le paramètre de la nouvelle porte)
    def cost_sequential(new_param):
        # A compléter : Concaténez les paramètres gelés et le nouveau paramètre
        # all_params = np.concatenate((fixed_params, new_param))
        
        # A compléter : Évaluez le circuit complet avec all_params via l'Estimator
        # job = estimator.run(...)
        # return job.result().values[0]
        pass
    
    # Le nouveau paramètre commence à 0
    initial_new_param = [0.0]
    
    # A compléter : Optimisez UNIQUEMENT le nouveau paramètre
    # res_seq = minimize(cost_sequential, initial_new_param, method='COBYLA')
    
    # A compléter : Mettez à jour la liste des paramètres fixes avec la valeur trouvée
    # new_optimized_val = res_seq.x
    # fixed_params = np.concatenate((fixed_params, new_optimized_val))
    
    # current_energy_seq = res_seq.fun
    # history_seq_adapt.append(current_energy_seq)
    # print(f"Énergie (Paramètres gelés) : {current_energy_seq:.4f}\n")

# Optionnel : Ajoutez ici un plot pour comparer history_adapt (standard) et history_seq_adapt (séquentiel)

## 2. Le compromis ultime : k-ADAPT-VQE avec Optimisation Séquentielle

Pour repousser encore plus loin l'efficacité, nous pouvons combiner les deux stratégies précédentes. À chaque itération, nous allons ajouter les $k$ meilleurs opérateurs (pour réduire le nombre de cycles globaux), mais nous n'allons optimiser **que les $k$ nouveaux paramètres** associés, en maintenant les paramètres des cycles précédents gelés.

Cette méthode "gloutonne groupée" est souvent un excellent compromis entre temps de calcul CPU (optimisation classique rapide) et profondeur du circuit quantique.

**Méthodologie détaillée :**

1. Calculez les gradients de tout le pool et isolez les indices des $k$ plus grands gradients (comme dans le k-ADAPT-VQE).
2. Ajoutez ces $k$ opérateurs au circuit.
3. Créez une fonction de coût qui accepte un vecteur de **$k$ nouveaux paramètres**. Elle doit les concaténer à la fin de la liste `fixed_params_k` pour évaluer le circuit complet.
4. Lancez l'optimisation sur ce petit vecteur de taille $k$.
5. Mettez à jour la liste des paramètres gelés avec ces $k$ nouvelles valeurs.
6. Tracez l'historique complet pour observer l'impact de ce gel paramétrique sur la convergence.

In [ ]:
k = 2
current_circuit_seq_k = init_state.copy()
history_seq_k_adapt = []
fixed_params_k = [] # Stockage des paramètres gelés

for iteration in range(20):
    print(f"--- Cycle Séq-{k}-ADAPT-VQE {iteration+1} ---")
    
    # 1. Évaluation des gradients
    gradients = []
    for op in operator_pool:
        # A compléter : calcul du gradient 
        grad_val = ... 
        gradients.append(np.abs(grad_val))
        
    # 2. Sélection des k meilleurs opérateurs
    # A compléter : trouver les k indices avec les plus grands gradients
    indices_tries = ... 
    top_k_indices = indices_tries[:k]
    
    if gradients[top_k_indices[0]] < gradient_threshold:
         print("Convergence atteinte.")
         break
         
    # 3. Ajout groupé au circuit
    for idx in top_k_indices:
        op_to_add = operator_pool[idx]
        # A compléter : ajouter l'opérateur au current_circuit_seq_k
        # ...
        
    # 4. Optimisation Séquentielle sur k paramètres
    def cost_sequential_k(new_k_params):
        # A compléter : Concaténez les paramètres gelés et les k nouveaux
        # all_params = np.concatenate((fixed_params_k, new_k_params))
        
        # A compléter : Évaluez le circuit complet via l'Estimator
        # job = estimator.run(...)
        # return job.result().values[0]
        pass
    
    # Les k nouveaux paramètres commencent à 0
    initial_new_params = np.zeros(k)
    
    # A compléter : Optimisez UNIQUEMENT les k nouveaux paramètres
    # res_seq_k = minimize(...) 
    
    # A compléter : Ajoutez les k valeurs optimisées à la liste des paramètres gelés
    # new_optimized_vals = res_seq_k.x
    # fixed_params_k = np.concatenate((fixed_params_k, new_optimized_vals))
    
    current_energy_seq_k = res_seq_k.fun
    history_seq_k_adapt.append(current_energy_seq_k)
    print(f"Énergie (Paramètres gelés, k={k}) : {current_energy_seq_k:.4f}\n")

## 3. Bilan final : Tracé de toutes les méthodes adaptatives

Nous allons maintenant superposer toutes nos approches pour déterminer laquelle offre le meilleur chemin vers l'énergie fondamentale de notre molécule.

* **ADAPT-VQE standard (k=1) :** Lent en cycles, optimisation classique lourde, mais trouve le meilleur chemin quantique.
* **k-ADAPT-VQE (k=2) :** Moins de cycles, optimisation classique lourde.
* **Séq-ADAPT-VQE (k=1, Gelé) :** Plus de cycles, optimisation ultra-légère (1 paramètre), risque de minimum local.
* **Séq-k-ADAPT-VQE (k=2, Gelé) :** Cycles réduits, optimisation légère (k paramètres).


In [ ]:
plt.figure(figsize=(12, 7))

# A compléter : dé-commentez et ajustez si besoin les noms de vos variables d'historique
# plt.plot(range(1, len(history_adapt) + 1), history_adapt, 
#          marker='o', linestyle='-', color='red', label='ADAPT-VQE (k=1, Ré-opt totale)')

# plt.plot(range(1, len(history_k_adapt) + 1), history_k_adapt, 
#          marker='s', linestyle='-', color='green', label=f'k-ADAPT-VQE (k={k}, Ré-opt totale)')

# plt.plot(range(1, len(history_seq_adapt) + 1), history_seq_adapt, 
#          marker='^', linestyle='--', color='orange', label='Séq-ADAPT-VQE (k=1, Paramètres gelés)')

# plt.plot(range(1, len(history_seq_k_adapt) + 1), history_seq_k_adapt, 
#          marker='d', linestyle='--', color='purple', label=f'Séq-k-ADAPT-VQE (k={k}, Paramètres gelés)')

plt.xlabel('Cycles Adaptatifs')
plt.ylabel('Énergie (Hartree)')
plt.title('Comparaison de convergence : Stratégies de ré-optimisation et Ajouts groupés')
plt.legend()
plt.grid(True)
plt.show()